# Dataset exploration

Inspect the source training and validation images before creating model-ready splits. The notebook expects the folder layout documented in the project README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

RAW_DIR = Path("data/raw")
CLASS_FOLDERS = {
    "01-minor": "Low",
    "02-moderate": "Medium",
    "03-severe": "High",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {RAW_DIR.resolve()}")

records = []
for split in ("training", "validation"):
    for folder, label in CLASS_FOLDERS.items():
        class_dir = RAW_DIR / split / folder
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Class directory not found: {class_dir.resolve()}")
        records.extend(
            {"split": split, "class": label, "path": path}
            for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )

images = pd.DataFrame(records)
if images.empty:
    raise ValueError(f"No supported images found under {RAW_DIR.resolve()}")
images.groupby(["split", "class"]).size().unstack(fill_value=0)

In [ ]:
counts = images.groupby(["split", "class"]).size().unstack(fill_value=0)
counts.plot(kind="bar", figsize=(9, 5), color=["#8bb3c2", "#527f96", "#d8b98c"])
plt.title("Image counts by split and severity")
plt.xlabel("Source split")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
sample = images.sample(min(9, len(images)), random_state=42)
figure, axes = plt.subplots(3, 3, figsize=(10, 9))
for axis, (_, row) in zip(axes.flat, sample.iterrows()):
    with Image.open(row["path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"{row['split']}: {row['class']}")
    axis.axis("off")
for axis in list(axes.flat)[len(sample):]:
    axis.axis("off")
figure.tight_layout()
plt.show()